# This notebook converts the .mat data to netcdf format, 
# add the necessary metadata and plot the data

## load library

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os, sys
import json

In [2]:
# Import modules
%reload_ext autoreload
%autoreload 2

# from src.netcdf import mat_to_xarray
sys.path.append('/Users/yugao/UOP/ORS-processing/src')
from metadata import create_json
from netcdf import read_mat_file, mat_to_xarray, save_to_netcdf

## INPUT REQUIRED: 
## Depth parameters from mooring diagram and mooring log 

In [3]:
instrument_height_above_bottom =  39   
water_depth_from_mooring_diagram = 4450
water_depth_from_ship_uncorrected = 4562.2      # uncorrected water depth 
water_depth_from_ship_corrected = 4538.97       # Replace with actual value, best water depth

#  Anchor (1) + chain (5) + nystron (20) + chain (5) + releases (1) + chain (5) 
# + 6 terminations at 0.25 ea (1.5) 
# + distance from termination on SBE cage to sensor (0.5) = 39 m

In [4]:
depth_parameters = {
    # consult mooring diagram
    'water_depth_from_mooring_diagram': water_depth_from_mooring_diagram,  
     # uncorrected water depth
    'water_depth_from_ship_uncorrected': water_depth_from_ship_uncorrected,
    # Replace with actual value, best water depth
    'water_depth_from_ship_corrected': water_depth_from_ship_corrected,        
    # instrument_depth_from_mooring_diagram = diagram depth  - height above bottom 
    'instrument_depth_from_mooring_diagram': water_depth_from_mooring_diagram - instrument_height_above_bottom,  
    # instrument_depth_from_mooring_log = corrected depth (4538.97 m) - height above bottom (39 m) 
    'instrument_depth_from_mooring_log': water_depth_from_ship_corrected - instrument_height_above_bottom,
    'instrument_height_above_bottom': 39     
    #  Anchor (1) + chain (5) + nystron (20) + chain (5) + releases (1) + chain (5) 
    # + 6 terminations at 0.25 ea (1.5) 
    # + distance from termination on SBE cage to sensor (0.5) = 39 m
}
depth_parameters 

{'water_depth_from_mooring_diagram': 4450,
 'water_depth_from_ship_uncorrected': 4562.2,
 'water_depth_from_ship_corrected': 4538.97,
 'instrument_depth_from_mooring_diagram': 4411,
 'instrument_depth_from_mooring_log': 4499.97,
 'instrument_height_above_bottom': 39}

## read mat file

In [5]:
mat_filepath = '../../data/external/stratus12_sbe16_1876.mat'
mat_data = read_mat_file(mat_filepath)
# Inspect the structure of mat_data
# print(mat_data.keys()) 

## convert mat file to netcdf

In [6]:
sbe16_metadata = mat_data['meta']

In [7]:
# Use the loaded MATLAB data (mat_data_updated) for conversion
ds = mat_to_xarray(mat_data)

## Meta data: we will add depth paramter for deep T/S

In [12]:
metadata_dict = create_json(mat_data, depth_parameters)
# Sample JSON file
json_path = '/Users/yugao/UOP/ORS-processing/data/metadata/stratus_OS_NTAS_2016_D_TS.json.json'

# Convert the dictionary to a JSON string and save it
with open(json_path, 'w') as f:
    json.dump(metadata_dict, f)

### read json file

In [14]:
with open(json_path, 'r') as f:
    metadata_from_json = json.load(f)

### incorporate the json file with all attributes

In [15]:
metadata_from_json

{'attributes': {'site': 'Stratus',
  'deployment': '13',
  'principal_investigator': 'Robert Weller',
  'institution': 'WHOI',
  'project': 'WHOI-UOP',
  'platform_type': 'surface mooring',
  'platform_year': 'Stratus Ocean Reference Station',
  'time_coverage_start': '2014-02-28T12:59:59Z',
  'time_coverage_end': '2015-04-27T03:29:59Z',
  'time_coverage_duration': 'P422D',
  'latitude_anchor_survey': -19.624523333333332,
  'longitude_anchor_survey': -84.95232333333334,
  'geospatial_lat_min': -19.624523333333332,
  'geospatial_lat_max': -19.624523333333332,
  'geospatial_lon_min': -84.95232333333334,
  'geospatial_lon_max': -84.95232333333334,
  'geospatial_vertical_min': 4504,
  'geospatial_vertical_max': 4504,
  'time_coverage_resolution': 'PT1H'},
 'depth parameters': {'water_depth_from_mooring_diagram': 4450,
  'water_depth_from_ship_uncorrected': 4562.2,
  'water_depth_from_ship_corrected': 4538.97,
  'instrument_depth_from_mooring_diagram': 4411,
  'instrument_depth_from_mooring

In [16]:
# Flatten the nested dictionaries within the 'attributes' attribute
attributes_flat = {}
for key, value in  metadata_from_json.items():
    if isinstance(value, dict):
        for sub_key, sub_value in value.items():
            attributes_flat[f'{sub_key}'] = sub_value
    else:
        attributes_flat[key] = value

# Update the attributes with the flattened dictionary
ds.attrs.update(attributes_flat)

In [17]:
ds.attrs

{'depth': 4504,
 'latitude': -19.624523333333332,
 'longitude': -84.95232333333334,
 'site': 'Stratus',
 'deployment': '13',
 'principal_investigator': 'Robert Weller',
 'institution': 'WHOI',
 'project': 'WHOI-UOP',
 'platform_type': 'surface mooring',
 'platform_year': 'Stratus Ocean Reference Station',
 'time_coverage_start': '2014-02-28T12:59:59Z',
 'time_coverage_end': '2015-04-27T03:29:59Z',
 'time_coverage_duration': 'P422D',
 'latitude_anchor_survey': -19.624523333333332,
 'longitude_anchor_survey': -84.95232333333334,
 'geospatial_lat_min': -19.624523333333332,
 'geospatial_lat_max': -19.624523333333332,
 'geospatial_lon_min': -84.95232333333334,
 'geospatial_lon_max': -84.95232333333334,
 'geospatial_vertical_min': 4504,
 'geospatial_vertical_max': 4504,
 'time_coverage_resolution': 'PT1H',
 'water_depth_from_mooring_diagram': 4450,
 'water_depth_from_ship_uncorrected': 4562.2,
 'water_depth_from_ship_corrected': 4538.97,
 'instrument_depth_from_mooring_diagram': 4411,
 'inst

In [18]:
ds

<xarray.Dataset> Size: 811kB
Dimensions:  (time: 20286)
Coordinates:
  * time     (time) float64 162kB 7.357e+05 7.357e+05 ... 7.361e+05 7.361e+05
Data variables:
    temp     (time) float64 162kB 8.968 4.617 17.52 20.0 ... 22.33 22.33 30.11
    cond     (time) float64 162kB -0.000225 -0.000225 ... 0.000333 6.084
    sal      (time) float64 162kB 0.003206 0.0001419 0.008013 ... 0.01125 35.47
    sal_sbe  (time) float64 162kB 0.0 0.0 0.0 0.0 ... 0.0113 0.0113 0.0113 35.47
Attributes: (12/28)
    depth:                                  4504
    latitude:                               -19.624523333333332
    longitude:                              -84.95232333333334
    site:                                   Stratus
    deployment:                             13
    principal_investigator:                 Robert Weller
    ...                                     ...
    water_depth_from_mooring_diagram:       4450
    water_depth_from_ship_uncorrected:      4562.2
    water_depth_from_ship_corrected:        4538.97
    instrument_depth_from_mooring_diagram:  4411
    instrument_depth_from_mooring_log:      4499.97
    instrument_height_above_bottom:         39

In [29]:
netcdf_path = '/Users/yugao/UOP/ORS-processing/data/processed/stratus/stratus' + mat_filepath[-17:-4] + '.nc'
netcdf_path 

'/Users/yugao/UOP/ORS-processing/data/processed/stratus/stratus13_sbe16_1873.nc'

In [30]:
# # Save the dataset to netCDF
ds.to_netcdf(netcdf_path)

In [31]:
ds.attrs

{'depth': 4504,
 'latitude': -19.624523333333332,
 'longitude': -84.95232333333334,
 'site': 'Stratus',
 'deployment': '13',
 'principal_investigator': 'Robert Weller',
 'institution': 'WHOI',
 'project': 'WHOI-UOP',
 'platform_type': 'surface mooring',
 'platform_year': 'Stratus Ocean Reference Station',
 'time_coverage_start': '2014-02-28T12:59:59Z',
 'time_coverage_end': '2015-04-27T03:29:59Z',
 'time_coverage_duration': 'P422D',
 'latitude_anchor_survey': -19.624523333333332,
 'longitude_anchor_survey': -84.95232333333334,
 'geospatial_lat_min': -19.624523333333332,
 'geospatial_lat_max': -19.624523333333332,
 'geospatial_lon_min': -84.95232333333334,
 'geospatial_lon_max': -84.95232333333334,
 'geospatial_vertical_min': 4504,
 'geospatial_vertical_max': 4504,
 'time_coverage_resolution': 'PT1H',
 'water_depth_from_mooring_diagram': 4450,
 'water_depth_from_ship_uncorrected': 4562.2,
 'water_depth_from_ship_corrected': 4538.97,
 'instrument_depth_from_mooring_diagram': 4411,
 'inst